# Calling APIs: Get Live Data from the Web 🌐

Here is something important about an LLM: **it cannot see the internet, and it does not know today's news.**

If you ask the LLM *"What is today's US dollar rate?"*, it will only guess — and the guess may be old or wrong. It does not check the latest rates every morning like we do.

So how do real apps get fresh, live data — today's weather, today's prices, today's news? They call an **API**.

And once your code can fetch live data, you can give that data to the LLM. **Live data + LLM = a real, useful app.** This is how AI assistants answer questions about *right now*.

## 1. What is an API?

An **API** is just a special web address (URL) that your code can call to get data back. You ask, it answers — and the answer usually comes as **JSON**.

Good news: you already know JSON from the last lessons. In Python it becomes a normal **dictionary**.

To call APIs we use a library called `requests`. If it is not installed, run this once:

```
pip install requests
```

Let's make our first call. We will use a free currency API (no signup, no key needed) to get today's US Dollar → Indian Rupee rate.

In [1]:
import requests

# This URL asks: "give me the latest rate from USD to INR"
url = "https://api.frankfurter.app/latest?from=USD&to=INR"

response = requests.get(url)

print(response.status_code)   # 200 means "OK, it worked"

200


`status_code` tells us if the call worked:
- **200** → success
- **404** → not found (wrong address)
- **500** → the server had a problem

Now let's read the actual data. `.json()` turns the answer into a Python dictionary:

In [2]:
data = response.json()

print(data)
print(type(data))   # it is just a dict!

{'amount': 1.0, 'base': 'USD', 'date': '2026-05-29', 'rates': {'INR': 95.0}}
<class 'dict'>


See? It is a normal dictionary. So we read it the same way we read any dict — with keys:

In [3]:
rate = data["rates"]["INR"]

print(f"Today, 1 US Dollar = {rate} Indian Rupees")

Today, 1 US Dollar = 95.0 Indian Rupees


## 2. Changing the request with parameters

We can ask for different currencies by changing the values we send. Instead of writing them inside the URL, we can pass them neatly using `params`:

In [4]:
response = requests.get(
    "https://api.frankfurter.app/latest",
    params={"from": "USD", "to": "EUR"}   # now USD to Euro
)

data = response.json()
print("1 USD =", data["rates"]["EUR"], "EUR")

1 USD = 0.85881 EUR


## 3. When things go wrong

The internet is not always reliable. Maybe there is no connection, or the address is wrong. We do not want our app to crash, so we use `try` / `except` (remember the error handling lesson):

In [5]:
try:
    response = requests.get("https://api.frankfurter.app/latest?from=USD&to=INR")

    if response.status_code == 200:
        data = response.json()
        print("Rate:", data["rates"]["INR"])
    else:
        print("Something went wrong. Status code:", response.status_code)

except Exception as e:
    print("Could not reach the API:", e)

Rate: 95.0


## 4. The payoff: give the LLM live data

Now the fun part. We fetch the live rate from the API, then ask the LLM to use it. Watch how we drop the live `rate` into the prompt:

In [6]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

# 1. get live data from the API
response = requests.get("https://api.frankfurter.app/latest?from=USD&to=INR")
rate = response.json()["rates"]["INR"]

# 2. give that data to the LLM
prompt = f'''
I have 200 NVIDIA stocks, today the price is 215$. 
How much money will I get in Indian rupees if I sell them all today.
Today's USD to INR conversion rate is {rate}
'''

try:
    answer = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    ).text
except Exception as e:
    answer = f"Something went wrong: {e}"

print(answer)

Let's break this down:

1.  **Total value in USD:**
    200 stocks * $215/stock = $43,000

2.  **Convert to Indian Rupees (INR):**
    $43,000 * 95.0 INR/USD = 4,085,000 INR

You will get **4,085,000 Indian Rupees** if you sell all your NVIDIA stocks today.


That is the whole idea: **fetch live data → put it in the prompt → let the LLM use it.**

The LLM did not *know* the rate. We looked it up for it. This simple pattern — let the AI use a tool to get real data — is the basic idea behind AI "agents." You just built the first version of one. 🎉

### A quick note on API keys

Our currency API was open and free. But many real APIs (weather, maps, payments) ask you to sign up and get a free **API key** first — just like our Gemini key.

The idea is the same. You usually send the key with your request, like this:

```python
requests.get(url, params={"api_key": "YOUR_KEY"})
```

or in the request headers. The website's documentation always tells you how.

## 🏋️ Exercise: A simple currency helper

1. Pick any two currencies (for example, EUR to INR).
2. Call the API and read the rate from the JSON.
3. Take an amount, say 100, and calculate how much it is in the other currency.
4. Print the result clearly, for example: `100 EUR = 9050 INR`.

**Bonus:** pass the converted amount to the LLM and ask it to write a one-line travel tip that mentions the amount. Now you have combined a live API with the LLM — a small but real app. 🚀

In [ ]:
# Your solution here
